In [1]:
import os
import re
import threading
import queue
import subprocess
import tkinter as tk
from tkinter import ttk, filedialog, messagebox, simpledialog

DEFAULT_USPEX_EXE = r"C:\Users\andre\Downloads\USPEX_25_windows\uspex.exe"


# -----------------------------
# INPUT.txt generators (3 modes)
# -----------------------------
def gen_fixed_composition(elements: list[str], counts: list[int]) -> str:
    # fixed-composition (left box)
    elems = " ".join(elements)
    nums = " ".join(str(x) for x in counts)
    return (
        "% atomType\n"
        f"{elems}\n"
        "% EndAtomType\n\n"
        "% numSpecies\n"
        f"{nums}\n"
        "% EndNumSpecies\n"
    )


def gen_single_block(elements: list[str], ratios: list[int], min_at: int, max_at: int) -> str:
    # single-block (center box)
    elems = " ".join(elements)
    nums = " ".join(str(x) for x in ratios)
    return (
        "% atomType\n"
        f"{elems}\n"
        "% EndAtomType\n\n"
        "% numSpecies\n"
        f"{nums}\n"
        "% EndNumSpecies\n\n"
        f"{min_at}    : minAt\n"
        f"{max_at}    : maxAt\n"
    )


def gen_variable_composition(elements: list[str], min_at: int, max_at: int) -> str:
    # variable-composition (right box)
    # numSpecies = identity matrix (each element can vary independently)
    elems = " ".join(elements)
    n = len(elements)
    lines = []
    for i in range(n):
        row = ["0"] * n
        row[i] = "1"
        lines.append(" ".join(row))
    matrix = "\n".join(lines)
    return (
        "% atomType\n"
        f"{elems}\n"
        "% EndAtomType\n\n"
        "% numSpecies\n"
        f"{matrix}\n"
        "% EndNumSpecies\n\n"
        f"{min_at}    : minAt\n"
        f"{max_at}    : maxAt\n"
    )


def parse_elements(s: str) -> list[str]:
    parts = [p.strip() for p in s.replace(",", " ").split()]
    return [p for p in parts if p]


def parse_int_list(s: str) -> list[int]:
    parts = [p.strip() for p in s.replace(",", " ").split()]
    out = []
    for p in parts:
        if not p:
            continue
        out.append(int(p))
    return out


# -----------------------------
# USPEX output parser / tracker
# -----------------------------
GEN_RE = re.compile(r"==\s*Generation\s+(\d+)\s+started\s*==")
PRELIM_SIZE_RE = re.compile(r"preliminary generation size:\s*(\d+)")
FREED_RE = re.compile(r"Freed cores \[.*?\] from job \(system\s+(\d+),\s*step\s+(\d+)\)")
STAGE_RELAX_RE = re.compile(r"\*\s*2\.\s*RELAXATION\s*\*")
STAGE_PROP_RE = re.compile(r"\*\s*3\.\s*PROPERTIES CALCULATION\s*\*")
RELAX_DONE_RE = re.compile(r"Parallel relaxation done for all structures\.", re.IGNORECASE)


class UspexMonitor:
    def __init__(self, on_update):
        self.on_update = on_update  # callback(dict)

        self.current_gen = None
        self.gen_size = None
        self.total_relax_steps = None  # gen_size * 3
        self.relax_seen: set[tuple[int, int]] = set()  # (system, step)
        self.stage = "idle"

    def reset_for_generation(self, gen: int):
        self.current_gen = gen
        self.gen_size = None
        self.total_relax_steps = None
        self.relax_seen.clear()
        self.stage = "generation"
        self._emit()

    def _emit(self):
        self.on_update(
            {
                "generation": self.current_gen,
                "gen_size": self.gen_size,
                "stage": self.stage,
                "done_steps": len(self.relax_seen),
                "total_steps": self.total_relax_steps,
            }
        )

    def feed_line(self, line: str):
        m = GEN_RE.search(line)
        if m:
            self.reset_for_generation(int(m.group(1)))
            return

        m = PRELIM_SIZE_RE.search(line)
        if m:
            self.gen_size = int(m.group(1))
            self.total_relax_steps = self.gen_size * 3
            self._emit()
            return

        if STAGE_RELAX_RE.search(line):
            self.stage = "relaxation"
            self._emit()
            return

        if RELAX_DONE_RE.search(line):
            # This marks end of relaxation stage for this generation
            self.stage = "relaxation_done"
            self._emit()
            return

        if STAGE_PROP_RE.search(line):
            self.stage = "properties"
            self._emit()
            return

        m = FREED_RE.search(line)
        if m:
            system = int(m.group(1))
            step = int(m.group(2))
            key = (system, step)
            if key not in self.relax_seen:
                self.relax_seen.add(key)
                # stay in relaxation if we're seeing freed lines
                if self.stage not in ("properties",):
                    self.stage = "relaxation"
                self._emit()
            return


# -----------------------------
# GUI App
# -----------------------------
class App(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("USPEX GUI Monitor")
        self.geometry("1100x720")

        self.proc: subprocess.Popen | None = None
        self.reader_thread: threading.Thread | None = None
        self.q = queue.Queue()
        self.monitor = UspexMonitor(self._on_monitor_update)

        # state
        self.mode = tk.StringVar(value="fixed")
        self.uspex_path = tk.StringVar(value=DEFAULT_USPEX_EXE)
        self.workdir = tk.StringVar(value="")

        self._build_ui()
        self._set_mode("fixed")
        self.after(100, self._poll_queue)

    # ---------- UI ----------
    def _build_ui(self):
        top = ttk.Frame(self, padding=10)
        top.pack(fill="x")

        # USPEX path
        ttk.Label(top, text="Путь к uspex.exe:").grid(row=0, column=0, sticky="w")
        e = ttk.Entry(top, textvariable=self.uspex_path, width=80)
        e.grid(row=0, column=1, sticky="we", padx=8)
        ttk.Button(top, text="Выбрать…", command=self._pick_exe).grid(row=0, column=2, sticky="e")

        # Workdir
        ttk.Label(top, text="Рабочая папка:").grid(row=1, column=0, sticky="w", pady=(8, 0))
        e2 = ttk.Entry(top, textvariable=self.workdir, width=80)
        e2.grid(row=1, column=1, sticky="we", padx=8, pady=(8, 0))
        ttk.Button(top, text="Выбрать…", command=self._pick_workdir).grid(row=1, column=2, sticky="e", pady=(8, 0))
        ttk.Button(top, text="Создать…", command=self._create_workdir).grid(row=1, column=3, sticky="e", padx=(8, 0), pady=(8, 0))

        top.columnconfigure(1, weight=1)

        mid = ttk.Frame(self, padding=(10, 0, 10, 10))
        mid.pack(fill="both", expand=True)

        # Left control panel
        left = ttk.Frame(mid)
        left.pack(side="left", fill="y", padx=(0, 10))

        ttk.Label(left, text="Режим INPUT.txt", font=("Segoe UI", 10, "bold")).pack(anchor="w", pady=(0, 6))
        ttk.Button(left, text="Знаю состав", command=lambda: self._set_mode("fixed")).pack(fill="x", pady=2)
        ttk.Button(left, text="Знаю соотношение", command=lambda: self._set_mode("single")).pack(fill="x", pady=2)
        ttk.Button(left, text="Знаю химические элементы", command=lambda: self._set_mode("variable")).pack(fill="x", pady=2)

        ttk.Separator(left, orient="horizontal").pack(fill="x", pady=10)

        self.form = ttk.LabelFrame(left, text="Параметры", padding=8)
        self.form.pack(fill="x")

        # form fields (created once, shown/hidden by mode)
        self.el_var = tk.StringVar(value="Si O")

        self.fixed_counts_var = tk.StringVar(value="6 12")

        self.single_ratios_var = tk.StringVar(value="1 2")
        self.single_min_var = tk.StringVar(value="8")
        self.single_max_var = tk.StringVar(value="18")

        self.var_min_var = tk.StringVar(value="8")
        self.var_max_var = tk.StringVar(value="18")

        # shared: elements
        ttk.Label(self.form, text="Элементы (через пробел):").grid(row=0, column=0, sticky="w")
        self.el_entry = ttk.Entry(self.form, textvariable=self.el_var, width=28)
        self.el_entry.grid(row=1, column=0, sticky="we", pady=(0, 8))

        # fixed fields
        self.fixed_lbl = ttk.Label(self.form, text="Число атомов (по элементам):")
        self.fixed_entry = ttk.Entry(self.form, textvariable=self.fixed_counts_var)

        # single fields
        self.single_lbl = ttk.Label(self.form, text="Соотношение (целые числа):")
        self.single_entry = ttk.Entry(self.form, textvariable=self.single_ratios_var)
        self.single_min_lbl = ttk.Label(self.form, text="minAt:")
        self.single_min_entry = ttk.Entry(self.form, textvariable=self.single_min_var, width=10)
        self.single_max_lbl = ttk.Label(self.form, text="maxAt:")
        self.single_max_entry = ttk.Entry(self.form, textvariable=self.single_max_var, width=10)

        # variable fields
        self.var_min_lbl = ttk.Label(self.form, text="minAt:")
        self.var_min_entry = ttk.Entry(self.form, textvariable=self.var_min_var, width=10)
        self.var_max_lbl = ttk.Label(self.form, text="maxAt:")
        self.var_max_entry = ttk.Entry(self.form, textvariable=self.var_max_var, width=10)

        self.form.columnconfigure(0, weight=1)

        ttk.Separator(left, orient="horizontal").pack(fill="x", pady=10)

        # Run controls
        self.run_btn = ttk.Button(left, text="Запустить расчёт", command=self._start)
        self.run_btn.pack(fill="x", pady=2)
        self.stop_btn = ttk.Button(left, text="Остановить", command=self._stop, state="disabled")
        self.stop_btn.pack(fill="x", pady=2)

        # Right side: preview + log + progress
        right = ttk.Frame(mid)
        right.pack(side="left", fill="both", expand=True)

        # Status panel
        status = ttk.Frame(right)
        status.pack(fill="x")

        self.status_text = tk.StringVar(value="Готово.")
        self.gen_text = tk.StringVar(value="Поколение: —")
        self.stage_text = tk.StringVar(value="Стадия: —")

        ttk.Label(status, textvariable=self.gen_text).pack(side="left", padx=(0, 12))
        ttk.Label(status, textvariable=self.stage_text).pack(side="left", padx=(0, 12))
        ttk.Label(status, textvariable=self.status_text).pack(side="left")

        # progress
        self.progress = ttk.Progressbar(right, mode="determinate")
        self.progress.pack(fill="x", pady=(6, 10))

        # Input preview
        prev_frame = ttk.LabelFrame(right, text="INPUT.txt (предпросмотр)", padding=8)
        prev_frame.pack(fill="both", expand=False)

        self.preview = tk.Text(prev_frame, height=10, wrap="none")
        self.preview.pack(fill="both", expand=True)
        self.preview.configure(state="disabled")

        # Log
        log_frame = ttk.LabelFrame(right, text="Лог USPEX", padding=8)
        log_frame.pack(fill="both", expand=True, pady=(10, 0))

        self.log = tk.Text(log_frame, wrap="none")
        self.log.pack(fill="both", expand=True)
        self.log.configure(state="disabled")

        # bind changes -> update preview
        for var in [
            self.mode, self.el_var, self.fixed_counts_var,
            self.single_ratios_var, self.single_min_var, self.single_max_var,
            self.var_min_var, self.var_max_var
        ]:
            var.trace_add("write", lambda *_: self._refresh_preview())

        self._refresh_preview()

    def _pick_exe(self):
        path = filedialog.askopenfilename(
            title="Выберите uspex.exe",
            filetypes=[("Executable", "*.exe"), ("All files", "*.*")]
        )
        if path:
            self.uspex_path.set(path)

    def _pick_workdir(self):
        path = filedialog.askdirectory(title="Выберите рабочую папку")
        if path:
            self.workdir.set(path)

    def _create_workdir(self):
        parent = filedialog.askdirectory(title="Выберите родительскую папку")
        if not parent:
            return
        name = simpledialog.askstring("Создать папку", "Имя новой папки:")
        if not name:
            return
        path = os.path.join(parent, name)
        try:
            os.makedirs(path, exist_ok=True)
        except Exception as e:
            messagebox.showerror("Ошибка", f"Не удалось создать папку:\n{e}")
            return
        self.workdir.set(path)

    def _set_mode(self, mode: str):
        self.mode.set(mode)

        # Clear form grid except common elements row 0-1 already placed
        # We'll (re)grid mode-specific widgets under it.
        for w in [
            self.fixed_lbl, self.fixed_entry,
            self.single_lbl, self.single_entry,
            self.single_min_lbl, self.single_min_entry, self.single_max_lbl, self.single_max_entry,
            self.var_min_lbl, self.var_min_entry, self.var_max_lbl, self.var_max_entry
        ]:
            w.grid_forget()

        r = 2  # start below elements entry
        if mode == "fixed":
            self.fixed_lbl.grid(row=r, column=0, sticky="w")
            self.fixed_entry.grid(row=r + 1, column=0, sticky="we", pady=(0, 8))
        elif mode == "single":
            self.single_lbl.grid(row=r, column=0, sticky="w")
            self.single_entry.grid(row=r + 1, column=0, sticky="we", pady=(0, 8))

            self.single_min_lbl.grid(row=r + 2, column=0, sticky="w")
            self.single_min_entry.grid(row=r + 3, column=0, sticky="w", pady=(0, 8))
            self.single_max_lbl.grid(row=r + 4, column=0, sticky="w")
            self.single_max_entry.grid(row=r + 5, column=0, sticky="w", pady=(0, 8))
        else:  # variable
            self.var_min_lbl.grid(row=r, column=0, sticky="w")
            self.var_min_entry.grid(row=r + 1, column=0, sticky="w", pady=(0, 8))
            self.var_max_lbl.grid(row=r + 2, column=0, sticky="w")
            self.var_max_entry.grid(row=r + 3, column=0, sticky="w", pady=(0, 8))

        self._refresh_preview()

    def _refresh_preview(self):
        try:
            text = self._build_input_text(validate=False)
        except Exception as e:
            text = f"(ошибка в параметрах: {e})"

        self.preview.configure(state="normal")
        self.preview.delete("1.0", "end")
        self.preview.insert("1.0", text)
        self.preview.configure(state="disabled")

    def _build_input_text(self, validate: bool = True) -> str:
        mode = self.mode.get()
        elements = parse_elements(self.el_var.get())
        if validate and not elements:
            raise ValueError("Введите хотя бы один химический элемент.")

        if mode == "fixed":
            counts = parse_int_list(self.fixed_counts_var.get())
            if validate and len(counts) != len(elements):
                raise ValueError("Количество чисел в 'Число атомов' должно совпадать с числом элементов.")
            return gen_fixed_composition(elements, counts)

        if mode == "single":
            ratios = parse_int_list(self.single_ratios_var.get())
            if validate and len(ratios) != len(elements):
                raise ValueError("Количество чисел в 'Соотношение' должно совпадать с числом элементов.")
            min_at = int(self.single_min_var.get())
            max_at = int(self.single_max_var.get())
            if validate and min_at <= 0:
                raise ValueError("minAt должен быть > 0.")
            if validate and max_at < min_at:
                raise ValueError("maxAt должен быть >= minAt.")
            return gen_single_block(elements, ratios, min_at, max_at)

        # variable
        min_at = int(self.var_min_var.get())
        max_at = int(self.var_max_var.get())
        if validate and min_at <= 0:
            raise ValueError("minAt должен быть > 0.")
        if validate and max_at < min_at:
            raise ValueError("maxAt должен быть >= minAt.")
        return gen_variable_composition(elements, min_at, max_at)

    # ---------- process control ----------
    def _start(self):
        if self.proc is not None:
            return

        exe = self.uspex_path.get().strip()
        wd = self.workdir.get().strip()

        if not exe or not os.path.isfile(exe):
            messagebox.showerror("Ошибка", "Путь к uspex.exe неверный.")
            return
        if not wd:
            messagebox.showerror("Ошибка", "Выберите рабочую папку.")
            return
        if not os.path.isdir(wd):
            try:
                os.makedirs(wd, exist_ok=True)
            except Exception as e:
                messagebox.showerror("Ошибка", f"Не удалось создать рабочую папку:\n{e}")
                return

        # Build and write INPUT.txt
        try:
            input_text = self._build_input_text(validate=True)
        except Exception as e:
            messagebox.showerror("Ошибка в INPUT", str(e))
            return

        try:
            with open(os.path.join(wd, "INPUT.txt"), "w", encoding="utf-8", newline="\n") as f:
                f.write(input_text)
        except Exception as e:
            messagebox.showerror("Ошибка", f"Не удалось записать INPUT.txt:\n{e}")
            return

        # Clear UI
        self._log_clear()
        self.progress["value"] = 0
        self.status_text.set("Запуск USPEX…")
        self.gen_text.set("Поколение: —")
        self.stage_text.set("Стадия: —")
        self.monitor = UspexMonitor(self._on_monitor_update)  # reset

        # Start process
        try:
            self.proc = subprocess.Popen(
                [exe],
                cwd=wd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
                encoding="utf-8",
                errors="replace",
            )
        except Exception as e:
            self.proc = None
            messagebox.showerror("Ошибка запуска", str(e))
            return

        self.run_btn.configure(state="disabled")
        self.stop_btn.configure(state="normal")

        self.reader_thread = threading.Thread(target=self._reader_loop, daemon=True)
        self.reader_thread.start()

    def _stop(self):
        if self.proc is None:
            return
        self.status_text.set("Остановка…")
        try:
            self.proc.terminate()
        except Exception:
            pass

    def _reader_loop(self):
        assert self.proc is not None
        try:
            for line in self.proc.stdout:
                self.q.put(("line", line.rstrip("\n")))
        except Exception as e:
            self.q.put(("err", str(e)))
        finally:
            code = None
            try:
                code = self.proc.wait(timeout=1)
            except Exception:
                pass
            self.q.put(("exit", code))

    def _poll_queue(self):
        try:
            while True:
                kind, payload = self.q.get_nowait()
                if kind == "line":
                    self._append_log(payload)
                    self.monitor.feed_line(payload)
                elif kind == "err":
                    self._append_log(f"[GUI] Ошибка чтения вывода: {payload}")
                elif kind == "exit":
                    self._append_log(f"[GUI] USPEX завершился (код: {payload})")
                    self.proc = None
                    self.run_btn.configure(state="normal")
                    self.stop_btn.configure(state="disabled")
                    self.status_text.set("Готово (процесс завершён).")
        except queue.Empty:
            pass
        self.after(100, self._poll_queue)

    # ---------- monitor callback ----------
    def _on_monitor_update(self, info: dict):
        gen = info.get("generation")
        stage = info.get("stage")
        gen_size = info.get("gen_size")
        done = info.get("done_steps")
        total = info.get("total_steps")

        self.gen_text.set(f"Поколение: {gen if gen is not None else '—'}")

        stage_map = {
            "idle": "—",
            "generation": "Создание поколения",
            "relaxation": "Релаксация",
            "relaxation_done": "Релаксация завершена",
            "properties": "Расчёт свойств",
        }
        self.stage_text.set(f"Стадия: {stage_map.get(stage, stage)}")

        if total and total > 0:
            self.progress["maximum"] = total
            self.progress["value"] = min(done, total)
            self.status_text.set(f"Релаксация: {done}/{total} шагов (структур: {gen_size or '—'})")
        else:
            self.progress["maximum"] = 1
            self.progress["value"] = 0
            if gen_size:
                self.status_text.set(f"Структур в поколении: {gen_size}. Ожидаю релаксацию…")

        if stage == "relaxation_done":
            self.status_text.set("Релаксация поколения завершена, идёт расчёт свойств…")

    # ---------- log helpers ----------
    def _log_clear(self):
        self.log.configure(state="normal")
        self.log.delete("1.0", "end")
        self.log.configure(state="disabled")

    def _append_log(self, line: str):
        self.log.configure(state="normal")
        self.log.insert("end", line + "\n")
        self.log.see("end")
        self.log.configure(state="disabled")


if __name__ == "__main__":
    App().mainloop()
